# Phase 2B — Provenance gate for a controlled BiLSTM experiment
Before constructing a new split, verify the seq20 feature schema, source/event/station identities, timestamps, target alignment and preprocessing provenance. Raw-catalog row order and a “station_grouped” filename do not establish them.

This notebook inspects only the BiLSTM artifacts, root preprocessing sources and named source catalogs. It never runs a dataset-generation script. Existing Phase 2 artifacts are preserved. If provenance is insufficient, it records a blocked experiment and performs only non-destructive descriptive checks; no model or baseline is fitted. Existing test diagnostics are not used to select preprocessing or model changes.

In [1]:
from pathlib import Path
import os,sys,json,hashlib,csv,zipfile,ast,itertools
ROOT=Path.cwd().resolve()
if (ROOT/"bilstm/bilstm_dataset").is_dir(): ROOT=ROOT/"bilstm"
assert (ROOT/"bilstm_dataset").is_dir()
PROJECT=ROOT.parent
if (ROOT/".notebook-runtime").exists(): sys.path.insert(0,str(ROOT/".notebook-runtime"))
import numpy as np
import pandas as pd
import torch
from IPython.display import display,Image
OUT=ROOT/"phase2b_generalization_results"
OUT.mkdir(exist_ok=True)
RES=ROOT/"phase2_bilstm_results"
def digest(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
def save_json(path,obj):
    tmp=Path(path).with_suffix(".tmp")
    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False),encoding="utf-8")
    tmp.replace(path)
source_paths=[PROJECT/n for n in ("preprocess_dl_temporal.py","preprocess.py","status.md",
                                  "dataset_2.csv","preprocessed_dl_temporal.csv")]
source_paths += [ROOT/"bilstm_dataset"/n for n in (
    "bilstm_dataset_seq20.pt","bilstm_dataset_seq20.npz","bilstm_dataset_seq20_station_grouped.npz")]
source_paths += [RES/n for n in ("generalization_audit/audit_report.md",
    "generalization_audit/audit_measurements.json","generalization_audit/verification.json",
    "test_metrics.json","target_transform.json","best_hyperparameters.json")]
protected=source_paths+[ROOT/"phase2_bilstm.ipynb",ROOT/"phase2_bilstm_generalization_audit.ipynb"]
protected += [RES/n for n in ("best_bilstm.pt","optuna_study.db","test_predictions.npz",
    "test_predictions.csv","training_history.csv","final_report.md")]
before={p.relative_to(PROJECT).as_posix():digest(p) for p in protected if p.exists()}
fingerprint=hashlib.sha256(json.dumps({"version":1,"files":before},sort_keys=True).encode()).hexdigest()
audit=json.loads((RES/"generalization_audit/audit_measurements.json").read_text())["confirmed"]
reference_metrics=json.loads((RES/"test_metrics.json").read_text())
assert before["bilstm/bilstm_dataset/bilstm_dataset_seq20.pt"]==reference_metrics["identity"]["dataset_sha256"]
FIXED={"hidden_size":64,"layers":2,"dropout":.1015,"lr":.002141,"weight_decay":6.04e-6,
       "batch_size":64,"optimizer":"AdamW","max_epochs":50,"patience":8,"gradient_clip":1.0,"seed":42}
cache_path=OUT/"provenance_evidence.json"
CACHE=cache_path.exists()
if CACHE:
    evidence=json.loads(cache_path.read_text())
    assert evidence["fingerprint"]==fingerprint, "Sources changed; preserve this result and reassess provenance."
print("Scope: named root sources and BiLSTM datasets only; cached evidence:",CACHE)


Scope: named root sources and BiLSTM datasets only; cached evidence: False


## Identity, source and numerical overlap checks
The raw catalog's IDs/timestamps are inventoried separately from seq20 windows; no mapping is manufactured. Archive headers are checked without loading alternate dataset arrays. The primary PT is loaded once if evidence is not cached.

Exact SHA-256 fingerprints compare complete windows, step vectors and contiguous 19-step blocks across splits. Matching vectors establish numerical reuse only; nonmatching vectors cannot prove event independence. Stored adjacency is measured only as a tensor property, never treated as chronology. Legacy 15×44 documentation is not transferred to the 20×36 dataset.

In [2]:
def array_hash(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()
def npz_headers(path):
    result={}
    with zipfile.ZipFile(path) as z:
        for item in z.infolist():
            with z.open(item) as f:
                v=np.lib.format.read_magic(f)
                if v==(1,0): shape,order,dtype=np.lib.format.read_array_header_1_0(f)
                elif v==(2,0): shape,order,dtype=np.lib.format.read_array_header_2_0(f)
                else: raise ValueError(f"Unreviewed NPY header version: {v}")
                result[item.filename.removesuffix(".npy")]={"shape":list(shape),"dtype":str(dtype)}
    return result

if not CACHE:
    # Targeted inventory, no recursive repository search.
    scope={}
    for label,folder in (("project_root",PROJECT),("bilstm_root",ROOT),("dataset_directory",ROOT/"bilstm_dataset")):
        scope[label]=sorted(p.name for p in folder.iterdir() if p.is_file()
                            and p.suffix.lower() in (".py",".ipynb",".json",".yaml",".yml",".csv",".pkl",".npz",".pt"))
    archive_headers={p.name:npz_headers(p) for p in (ROOT/"bilstm_dataset").glob("*.npz")}
    d=torch.load(ROOT/"bilstm_dataset/bilstm_dataset_seq20.pt",map_location="cpu",weights_only=True)
    primary_schema={k:{"shape":list(v.shape),"dtype":str(v.dtype)} if torch.is_tensor(v)
                    else {"type":type(v).__name__} for k,v in d.items()}
    assert all(tuple(d[f"X_{s}"].shape[1:])==(20,36) for s in ("train","val","test"))
    expected={f"{kind}_{s}" for s in ("train","val","test") for kind in ("X","y")}
    expected.update(f"y_{s}_{kind}" for s in ("train","val","test") for kind in ("raw","bin"))
    assert set(d)==expected, "Dataset schema changed: review newly available metadata before proceeding."
    arrays={s:d[f"X_{s}"].numpy() for s in ("train","val","test")}
    full={};rows={};blocks={};tokens={};adjacency={}
    for s,a in arrays.items():
        assert np.isfinite(a).all()
        full[s]={array_hash(w) for w in a}
        tokens[s]=[tuple(array_hash(v) for v in w) for w in a]
        rows[s]=set(itertools.chain.from_iterable(tokens[s]))
        blocks[s]={part for t in tokens[s] for part in (t[:19],t[1:])}
        adjacency[s]={"n_windows":len(a),"unique_window_values":len(full[s]),
                      "unique_step_vectors":len(rows[s]),
                      "adjacent_19step_matches":sum(tokens[s][i][1:]==tokens[s][i+1][:19] for i in range(len(a)-1)),
                      "interpretation":"Stored tensor adjacency only, not chronological evidence."}
    overlap=[]
    for a,b in itertools.combinations(arrays,2):
        shared=rows[a]&rows[b]
        overlap.append({"split_a":a,"split_b":b,
                        "shared_unique_full_windows":len(full[a]&full[b]),
                        "shared_unique_step_vectors":len(shared),
                        "shared_unique_contiguous_19step_blocks":len(blocks[a]&blocks[b]),
                        "a_windows_containing_shared_vectors":sum(any(v in shared for v in t) for t in tokens[a]),
                        "b_windows_containing_shared_vectors":sum(any(v in shared for v in t) for t in tokens[b])})
    target_check={}
    transform=json.loads((RES/"target_transform.json").read_text())
    for s in arrays:
        raw=d[f"y_{s}_raw"].numpy().reshape(-1).astype(np.float64)
        scaled=d[f"y_{s}"].numpy().reshape(-1).astype(np.float64)
        target_check[s]={"n":len(raw),"mean":float(raw.mean()),"median":float(np.median(raw)),
                         "min":float(raw.min()),"max":float(raw.max()),
                         "proportion_M_ge_4_5":float(np.mean(raw>=4.5)),
                         "inverse_max_error":float(np.max(np.abs(raw-(scaled*transform["scale"]+transform["offset"]))))}
    del d,arrays,tokens,full,rows,blocks

    # Read only necessary source-catalog metadata once. Preserve the literal ID strings.
    columns=["origin_time_utc","source_id","receiver_code","trace_name","Magnitude_Value"]
    catalog=pd.read_csv(PROJECT/"dataset_2.csv",usecols=columns,dtype="string")
    parsed=pd.to_datetime(catalog.origin_time_utc.str.replace(" UTC","",regex=False),format="mixed",errors="coerce",utc=True)
    def present(s):return s.notna() & s.fillna("").str.strip().ne("")
    sid=present(catalog.source_id);station=present(catalog.receiver_code);trace=present(catalog.trace_name)
    identities=catalog.assign(catalog_row_index=np.arange(len(catalog)),parsed_origin_time_utc=parsed.astype("string"))
    identities["seq20_mapping_status"]="UNAVAILABLE; catalog row is not a seq20 sample/event mapping"
    identities.to_csv(OUT/"source_catalog_identity_inventory.csv",index=False)
    grouped=identities.loc[sid].groupby("source_id",dropna=False)
    catalog_summary={"rows":len(catalog),"columns_inspected":columns,
        "valid_timestamp_rows":int(parsed.notna().sum()),
        "timestamp_min":parsed.min().isoformat() if parsed.notna().any() else None,
        "timestamp_max":parsed.max().isoformat() if parsed.notna().any() else None,
        "nonempty_source_id_rows":int(sid.sum()),"unique_source_ids":int(catalog.loc[sid,"source_id"].nunique()),
        "nonempty_station_rows":int(station.sum()),"unique_station_codes":int(catalog.loc[station,"receiver_code"].nunique()),
        "nonempty_trace_rows":int(trace.sum()),"unique_trace_names":int(catalog.loc[trace,"trace_name"].nunique()),
        "source_ids_with_multiple_timestamps":int((grouped.parsed_origin_time_utc.nunique()>1).sum()),
        "source_ids_with_multiple_magnitudes":int((grouped.Magnitude_Value.nunique()>1).sum()),
        "link_to_seq20":"UNKNOWN; no persisted correspondence found"}
    with (PROJECT/"preprocessed_dl_temporal.csv").open(newline="",encoding="utf-8-sig") as f:
        legacy_header=next(csv.reader(f))
    source_evidence={}
    needles=("sort_values(","quantile(0.99)",".median()","fit_transform(","Magnitude_Value",
             "depth_mag_ratio","log_energy","to_csv(","pickle.dump")
    for name in ("preprocess_dl_temporal.py","preprocess.py"):
        lines=(PROJECT/name).read_text(encoding="utf-8-sig").splitlines()
        source_evidence[name]=[{"line":i,"text":line.strip()} for i,line in enumerate(lines,1)
                              if any(n in line for n in needles)]
    legacy_text=(PROJECT/"preprocess_dl_temporal.py").read_text(encoding="utf-8-sig")
    tree=ast.parse(legacy_text)
    numeric_features=None
    for node in ast.walk(tree):
        if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id=="numeric_features" for t in node.targets):
            numeric_features=ast.literal_eval(node.value)
    root_status=(PROJECT/"status.md").read_text(encoding="utf-8-sig").splitlines()
    legacy_description=[{"line":i,"text":line.strip()} for i,line in enumerate(root_status,1)
                        if 150<=i<=170 and ("15" in line or "44" in line or "target" in line)]
    missing_named=[name for name in ("dl_temporal_scaler.pkl","bilstm/bilstm_temporal_model.py","bilstm/bilstm_v2_model.py")
                   if not (PROJECT/name).exists()]
    evidence={"fingerprint":fingerprint,"search_scope":scope,"source_hashes":before,
              "primary_schema":primary_schema,"archive_headers":archive_headers,
              "source_catalog":catalog_summary,"legacy_feature_header":legacy_header,
              "legacy_numeric_features":numeric_features,"legacy_source_excerpts":source_evidence,
              "legacy_sequence_description":legacy_description,"missing_legacy_files":missing_named,
              "target_checks":target_check,"within_split_numeric_adjacency":adjacency,
              "cross_split_numeric_overlap":overlap,
              "caveat":"Legacy 15x44 code/documentation is not verified as the source of primary seq20 tensors."}
    save_json(cache_path,evidence)
else:
    assert (OUT/"source_catalog_identity_inventory.csv").exists(), "Cached identity inventory missing."
display(pd.DataFrame(evidence["cross_split_numeric_overlap"]))
print(json.dumps({"catalog":evidence["source_catalog"],
                  "primary_keys":len(evidence["primary_schema"]),
                  "legacy_feature_count":len(evidence["legacy_feature_header"])},indent=2))


,split_a,split_b,shared_unique_full_windows,shared_unique_step_vectors,shared_unique_contiguous_19step_blocks,a_windows_containing_shared_vectors,b_windows_containing_shared_vectors
0,train,val,0,0,0,0,0
1,train,test,0,0,0,0,0
2,val,test,0,0,0,0,0


{
  "catalog": {
    "rows": 8089,
    "columns_inspected": [
      "origin_time_utc",
      "source_id",
      "receiver_code",
      "trace_name",
      "Magnitude_Value"
    ],
    "valid_timestamp_rows": 8089,
    "timestamp_min": "1906-08-31T14:57:30+00:00",
    "timestamp_max": "2026-10-06T09:03:32+00:00",
    "nonempty_source_id_rows": 8089,
    "unique_source_ids": 8089,
    "nonempty_station_rows": 8089,
    "unique_station_codes": 10,
    "nonempty_trace_rows": 8089,
    "unique_trace_names": 8089,
    "source_ids_with_multiple_timestamps": 0,
    "source_ids_with_multiple_magnitudes": 0,
    "link_to_seq20": "UNKNOWN; no persisted correspondence found"
  },
  "primary_keys": 12,
  "legacy_feature_count": 44
}


## PASS / FAIL / UNKNOWN and the training gate
PASS means the narrowly stated check is evidenced; FAIL means that check or prerequisite fails; UNKNOWN means the available evidence cannot resolve it. A failure in the legacy preprocessing code is not attributed to seq20 without lineage.

To train, the experiment requires a verified ordered feature schema, per-window input and target identities/times, target definition, exact builder and training-only preprocessing lineage. All must be verified before selecting a new split. Absence of these prerequisites blocks both BiLSTM and new-partition baseline fits.

In [3]:
checks=[]
def check(name,status,scope,detail):
    checks.append({"check":name,"status":status,"scope":scope,"evidence":detail})
check("Input tensor shape","PASS","seq20 primary","Existing inputs have 20 steps and 36 unnamed features; this establishes dimensions only.")
check("Exact seq20 builder available","FAIL","inspected scope",
      "No exact seq20 constructor is present in the inspected root/BiLSTM sources; legacy documentation describes 15x44.")
check("Meaning of the 20 steps","UNKNOWN","seq20 primary",
      "No saved step semantics or exact construction code; do not infer events or elapsed time from the sequence axis.")
check("Ordered meanings of the 36 features","UNKNOWN","seq20 primary",
      f"No schema in PT/NPZ artifacts. The available legacy CSV has {len(evidence['legacy_feature_header'])} columns, not 36.")
cs=evidence["source_catalog"]
check("Raw-catalog timestamp and identity columns","PASS","dataset_2.csv",
      f"{cs['valid_timestamp_rows']}/{cs['rows']} parseable UTC times; "
      f"{cs['nonempty_source_id_rows']} nonempty source IDs; {cs['nonempty_station_rows']} station codes. "
      "Existence in a candidate source is not a mapping to seq20.")
check("Source-ID timestamp consistency","PASS" if cs["source_ids_with_multiple_timestamps"]==0 else "FAIL",
      "raw-catalog internal check",
      f"{cs['source_ids_with_multiple_timestamps']} source IDs have multiple parsed origin times; identity semantics remain unverified.")
check("Window-to-source/event/time mapping available","FAIL","seq20 primary and alternate archive",
      "Both archives contain only X/y/scaled/raw/binary arrays. No input-row IDs, target IDs, times or station mapping.")
check("Inputs strictly precede target and are available at prediction time","UNKNOWN","seq20 primary",
      "No target horizon/alignment or feature availability times. Legacy magnitude-derived fields are a conditional leakage concern, not proof about seq20.")
check("Original split construction known","UNKNOWN","seq20 primary",
      "Saved partitions exist, but no cutoff/group/boundary manifest or verified constructor.")
check("Original preprocessing fitted only on training data","UNKNOWN","seq20 primary",
      "Correct target inversion does not establish input scaling/imputation/encoding fit provenance.")
check("Legacy preprocessing is training-only","FAIL","preprocess_dl_temporal.py only",
      "Global gap quantile (line 74), medians (line 134) and MinMax fit_transform (line 153) occur before any partitioning; seq20 linkage is unverified.")
check("Original target numeric inverse","PASS","seq20 supplied pairs",
      f"Maximum saved affine inverse error {max(v['inverse_max_error'] for v in evidence['target_checks'].values()):.3g}; timing/semantic target identity remains unknown.")
full_shared=sum(p["shared_unique_full_windows"] for p in evidence["cross_split_numeric_overlap"])
step_shared=sum(p["shared_unique_step_vectors"] for p in evidence["cross_split_numeric_overlap"])
check("No exact full-window duplicates across splits","PASS" if full_shared==0 else "FAIL","numerical equality only",
      f"{full_shared} shared unique full-window values across the three split-pair comparisons; this is not an event-independence test.")
check("No identical step vectors across splits","PASS" if step_shared==0 else "FAIL","numerical equality only",
      f"{step_shared} shared unique step-vector values summed across split pairs; identical values are not necessarily event IDs.")
check("Event-disjoint / overlap-purged split established","UNKNOWN","seq20 primary",
      "Numerical comparisons cannot certify shared events with different stations/representations, or prove overlap was purged.")
check("Chronological/event-disjoint resplit ready","FAIL","Phase 2B prerequisite",
      "Raw IDs/times exist but cannot be assigned to seq20 windows/targets. Repartitioning by stored row order would fabricate provenance.")

required_missing=[
    "Exact seq20 construction code and dataset/source version or hashes.",
    "Ordered 36-feature schema, units, derivations and feature availability times.",
    "For every window: original input row/event/station IDs and times for all 20 steps, plus target event ID and time.",
    "Original regression target definition, alignment/horizon and associated exclusions/filtering.",
    "Original split/group/cutoff and window-overlap/purge policy.",
    "Preprocessing fit rows/IDs and fitted scaling, imputation and categorical-encoding parameters."]
TRAINING_ALLOWED=False
assert any(c["status"]=="FAIL" and c["check"]=="Window-to-source/event/time mapping available" for c in checks)
save_json(OUT/"provenance_checks.json",checks)
save_json(OUT/"experiment_plan.json",{
    "status":"BLOCKED_PROVENANCE","training_allowed":TRAINING_ALLOWED,
    "fixed_settings_if_unblocked":FIXED,"target_policy":"Preserve original scaled-magnitude target and verify its event alignment before resplitting.",
    "required_information":required_missing,
    "future_split_policy":"With verified lineage, assign events chronologically, keep all station records together, purge boundary windows sharing events across partitions, and record discarded samples. Do not infer chronology from tensor order.",
    "future_preprocessing_policy":"Fit imputation, encoding and feature/target transformations on training records only; apply frozen transforms to validation and a new untouched holdout.",
    "evaluation_policy":"Validation-only early stopping; freeze checkpoint before holdout evaluation. Fit training-mean and validation-selected Ridge to the same new training partition. Different test populations are not proof of improvement.",
    "new_split_created":False,"new_scalers_fitted":False,"new_model_trained":False,"optuna_runs":0})
save_json(OUT/"preprocessing_assessment.json",{
    "seq20_train_only_fit":"UNKNOWN","legacy_global_fit":"FAIL",
    "legacy_link_to_seq20":"UNKNOWN","new_parameters_fitted":False,
    "known_target_inverse":json.loads((RES/"target_transform.json").read_text()),
    "note":"Known inverse coefficients are existing Phase 2 metadata, not newly fitted Phase 2B preprocessing."})
display(pd.DataFrame(checks)[["check","status","scope"]])
print("Training gate: BLOCKED. No new split, scaler, baseline, checkpoint or holdout evaluation.")


,check,status,scope
0,Input tensor shape,PASS,seq20 primary
1,Exact seq20 builder available,FAIL,inspected scope
2,Meaning of the 20 steps,UNKNOWN,seq20 primary
3,Ordered meanings of the 36 features,UNKNOWN,seq20 primary
4,Raw-catalog timestamp and identity columns,PASS,dataset_2.csv
5,Source-ID timestamp consistency,PASS,raw-catalog internal check
6,Window-to-source/event/time mapping available,FAIL,seq20 primary and alternate archive
7,Inputs strictly precede target and are availab...,UNKNOWN,seq20 primary
8,Original split construction known,UNKNOWN,seq20 primary
9,Original preprocessing fitted only on training...,UNKNOWN,seq20 primary


Training gate: BLOCKED. No new split, scaler, baseline, checkpoint or holdout evaluation.


In [4]:
table=["| Check | Status | Scope / evidence |","|---|---|---|"]
for c in checks:
    table.append(f"| {c['check']} | {c['status']} | {c['scope']}: {c['evidence']} |")
overlap_lines=["| Split pair | Exact full windows | Shared step vectors | Shared contiguous 19-step blocks |",
               "|---|---:|---:|---:|"]
for r in evidence["cross_split_numeric_overlap"]:
    overlap_lines.append(f"| {r['split_a']} / {r['split_b']} | {r['shared_unique_full_windows']} | "
                         f"{r['shared_unique_step_vectors']} | {r['shared_unique_contiguous_19step_blocks']} |")
distribution_lines=["| Existing partition | Samples | Mean magnitude | Median | M≥4.5 |",
                    "|---|---:|---:|---:|---:|"]
for s,r in evidence["target_checks"].items():
    distribution_lines.append(f"| {s} | {r['n']} | {r['mean']:.4f} | {r['median']:.2f} | {r['proportion_M_ge_4_5']:.1%} |")
provenance_report=f"""# Phase 2B split and provenance assessment

**Decision: BLOCKED before splitting or training.** The experiment prerequisites are not met. Existing tensors and artifacts are unchanged.

{chr(10).join(table)}

## Available source evidence
The candidate raw catalog contains {cs['rows']} rows, {cs['valid_timestamp_rows']} parseable UTC timestamps, {cs['unique_source_ids']} distinct nonempty source IDs and {cs['unique_station_codes']} distinct station codes. Timestamp extent: {cs['timestamp_min']} to {cs['timestamp_max']}. These are catalog properties, not confirmed times/IDs of seq20 inputs or targets.
The saved source_catalog_identity_inventory.csv preserves these candidate identities and explicitly labels their seq20 mapping unavailable. Its row index is a source-catalog index, not a sample ID or chronological split.

Root preprocess_dl_temporal.py sorts catalog dates (line 44), engineers temporal and magnitude-derived fields (including depth_mag_ratio/log_energy), then globally fits clipping/imputation/scaling before export. The CSV header contains {len(evidence['legacy_feature_header'])} columns; root status.md lines 154–165 describe 15-step, 44-feature next-event binary sequences. Neither establishes the lineage or causal alignment of the present 20×36 regression tensors.
The named legacy scaler and model files are absent: {', '.join(evidence['missing_legacy_files'])}. No generation script was executed. See provenance_evidence.json for exact source excerpts, line numbers, hashes and the bounded search inventory.

## Exact numerical overlap
{chr(10).join(overlap_lines)}

Counts describe identical float32 tensor values, not unique physical events. Shared numerical values do not identify events; no exact matches would not prove event independence. Adjacent-window 19-step equality is separately recorded as a storage property. No chronology is inferred from row ordering.
The station_grouped archive also lacks IDs, timestamps and a feature schema, so its name does not establish event-disjoint or chronological construction.

## Existing target distributions (descriptive only)
{chr(10).join(distribution_lines)}

These are the original populations, not new splits. Previously observed distribution shift and late overfitting remain distinct from unresolved preprocessing/event leakage. Out-of-training-range values alone do not prove a scaler error. The saved target inverse is numerically correct, but its upstream fitting lineage is not verified.

## Minimum evidence needed to unblock
{chr(10).join('- '+x for x in required_missing)}

Only after these prerequisites pass should an event-grouped chronological partition be constructed, with overlapping windows handled explicitly and training-only preprocessing saved. Fixed settings are recorded in experiment_plan.json. No parameter, clipping threshold or split was selected from the existing test results.
"""
(OUT/"split_provenance_report.md").write_text(provenance_report,encoding="utf-8")
save_json(OUT/"metrics.json",{
    "status":"NOT_RUN_PROVENANCE_BLOCKED",
    "new_experiment":{"train_mae":None,"train_rmse":None,"validation_mae":None,"validation_rmse":None,
        "holdout_mae":None,"holdout_rmse":None,"holdout_r2":None,"errors_by_magnitude":None,
        "underprediction_M_ge_4_5":None,"out_of_range_window_fraction":None,
        "in_range_mae":None,"out_of_range_mae":None,"training_mean_baseline":None,"ridge_baseline":None},
    "historical_phase2_reference_only":{
        "train_mae":audit["overfitting"]["tuned"]["best_train_mae"],
        "train_rmse":audit["overfitting"]["tuned"]["best_train_rmse"],
        "validation_mae":audit["overfitting"]["tuned"]["best_validation_mae"],
        "validation_rmse":audit["overfitting"]["tuned"]["best_validation_rmse"],
        "test":reference_metrics["tuned"]},
    "performance_change":"Not measured; no valid new experiment.",
    "original_results_used_for_selection":False})
final=f"""# Phase 2B — Generalization experiment decision

**Provenance investigation completed; controlled training blocked.** No new split, preprocessing fit, baseline fit, BiLSTM checkpoint or holdout evaluation was created.

| Question | Verdict |
|---|---|
| Chronological/event-disjoint splitting possible now? | **Not established.** Catalog timestamps/source IDs/station codes exist, but no verified mapping joins them to the 20-step windows and their targets. |
| Is preprocessing leakage-free? | **UNKNOWN for seq20.** The available legacy 44-feature script fits transforms globally (**FAIL** for that script); its relationship to seq20 is unverified. |
| Do inputs precede the target? | **UNKNOWN.** Step semantics, feature meanings/availability and target alignment are missing. |
| Valid controlled experiment completed? | **No.** The provenance gate failed before any splitting or training. |
| Generalization improvement? | **Not measured.** Historical validation MAE 0.2718 and test MAE/RMSE/R² 0.4341/0.6556/−0.0043 remain unchanged; they are not Phase 2B results. |
| Remaining limitations? | Missing exact constructor, ordered 36-feature schema, per-step and target identity/time mapping, split/purge policy, and preprocessing fit records. |
| Phase 3 scientifically feasible now? | **Not established.** It requires verified sample/event alignment, compatible causal targets and common evaluation partitions across modalities. No Phase 3 or Phase 4 work began. |

## New descriptive evidence
The raw catalog has {cs['valid_timestamp_rows']}/{cs['rows']} parseable origin timestamps and {cs['unique_source_ids']} distinct nonempty source IDs. The primary and station_grouped archives have only X/y arrays, with no identity/time/feature manifests. The available legacy documentation is 15×44, not 20×36.
Across the three existing split pairs, exact numerical checks found {full_shared} shared full-window values and {step_shared} shared step-vector values (pairwise sums, not event counts). Numerical equality checks cannot certify event independence or chronology.
Existing evidence supports distribution shift and late overfitting; neither establishes the cause of the test deterioration or rules out leakage.

## Next action
Recover the exact seq20 builder and its per-window input/target IDs, timestamps, feature schema and preprocessing fit manifest. Then validate a reproducible event-grouped chronological split with boundary-overlap handling and training-only transforms. Keep the requested BiLSTM settings fixed, select the checkpoint on validation, and reserve a new untouched holdout. Different holdout populations must not be treated as direct proof of improvement.

## Saved evidence
- [Split/provenance report](split_provenance_report.md), provenance_checks.json and provenance_evidence.json: PASS/FAIL/UNKNOWN findings, source references, schemas, overlap checks and hashes.
- source_catalog_identity_inventory.csv: original catalog identities only; explicitly **not** a seq20 split manifest.
- experiment_plan.json and preprocessing_assessment.json: fixed settings and blockers; **no newly fitted scaler parameters**.
- metrics.json: new-experiment fields are null and historical values are labeled separately.
- [Executed notebook](../phase2b_generalization_experiment.ipynb).
Existing [distribution figure](../phase2_bilstm_results/generalization_audit/magnitude_distributions.png) and [training curves](../phase2_bilstm_results/training_validation_curves.png) are referenced without generating duplicate or fictitious new-experiment plots.
"""
(OUT/"final_report.md").write_text(final,encoding="utf-8")
after={p.relative_to(PROJECT).as_posix():digest(p) for p in protected if p.exists()}
assert before==after, "An original artifact changed."
assert not list(OUT.glob("*.pt")), "A checkpoint must not exist for the blocked experiment."
save_json(OUT/"verification.json",{
    "status":"PROVENANCE_REVIEW_COMPLETE_EXPERIMENT_BLOCKED",
    "protected_artifacts_unchanged":True,"protected_sha256":after,
    "dataset_deserializations_this_run":0 if CACHE else 1,
    "alternate_datasets_header_only":True,"source_generation_scripts_executed":0,
    "bilstm_training_calls":0,"new_baseline_fits":0,"optuna_runs":0,
    "new_split_created":False,"new_preprocessing_fitted":False,
    "new_holdout_evaluations":0,"new_checkpoint_created":False,
    "notebook_final_cell_reached":True,"cache_fingerprint":fingerprint})
print(json.dumps({"outcome":"BLOCKED_PROVENANCE","checks":{s:sum(c["status"]==s for c in checks)
                  for s in ("PASS","FAIL","UNKNOWN")},"original_artifacts_unchanged":True,
                  "report":str(OUT/"final_report.md")},indent=2))


{
  "outcome": "BLOCKED_PROVENANCE",
  "checks": {
    "PASS": 6,
    "FAIL": 4,
    "UNKNOWN": 6
  },
  "original_artifacts_unchanged": true,
  "report": "C:\\Users\\choud\\Desktop\\Minor project\\bilstm\\phase2b_generalization_results\\final_report.md"
}
